results by category

In [1]:
import duckdb
con = duckdb.connect("../data/processed/retail.duckdb", read_only=True)

In [2]:
con.sql("""
    SELECT split_part(item_id, '_', 1)                       AS category,
           COUNT(DISTINCT store_id || '/' || item_id)        AS products,
           ROUND(100.0 * SUM(lost_sales) / SUM(demand), 2)   AS pct_demand_lost,
           ROUND(100.0 * SUM(CASE WHEN lost_sales > 0 THEN 1 ELSE 0 END)
                       / SUM(CASE WHEN on_sale THEN 1 ELSE 0 END), 2) AS pct_stockout_days,
           ROUND(AVG(end_stock) FILTER (WHERE on_sale), 1)   AS avg_shelf,
           SUM(expired)                                      AS expired_units,
           SUM(sold)                                         AS units_sold
    FROM '../data/processed/sim/daily/*/*.parquet'
    GROUP BY 1
    ORDER BY 1
""")

┌───────────┬──────────┬─────────────────┬───────────────────┬───────────┬───────────────┬────────────┐
│ category  │ products │ pct_demand_lost │ pct_stockout_days │ avg_shelf │ expired_units │ units_sold │
│  varchar  │  int64   │     double      │      double       │  double   │    int128     │   int128   │
├───────────┼──────────┼─────────────────┼───────────────────┼───────────┼───────────────┼────────────┤
│ FOODS     │     4311 │            5.05 │              1.65 │      12.1 │        319085 │   12181031 │
│ HOBBIES   │     1695 │            15.7 │              3.63 │       9.7 │             0 │    1683644 │
│ HOUSEHOLD │     3141 │             5.5 │              0.91 │      21.3 │             0 │    3742149 │
└───────────┴──────────┴─────────────────┴───────────────────┴───────────┴───────────────┴────────────┘

This table covers the development periods only. The live test stays out on purpose.

In [3]:
import duckdb, yaml

with open("../config.yaml") as f:
    cfg = yaml.safe_load(f)
P = cfg["periods"]

con = duckdb.connect()
sql = f"""
WITH d AS (
    SELECT
        split_part(item_id, '_', 1) AS category,
        CASE
            WHEN date BETWEEN '{P['burn_in'][0]}' AND '{P['burn_in'][1]}' THEN '1_burn_in'
            WHEN date BETWEEN '{P['block_a'][0]}' AND '{P['block_a'][1]}' THEN '2_block_a'
            WHEN date BETWEEN '{P['block_b'][0]}' AND '{P['block_b'][1]}' THEN '3_block_b'
        END AS period,
        demand, lost_sales
    FROM read_parquet('../data/processed/sim/daily/*/*.parquet', hive_partitioning=false)
    WHERE on_sale
)
SELECT category, period,
       round(100.0 * sum(lost_sales) / sum(demand), 2)                AS pct_demand_lost,
       round(100.0 * avg((lost_sales > 0)::int), 2)                   AS pct_stockout_days,
       round(sum(lost_sales) * 1.0 / nullif(sum((lost_sales > 0)::int), 0), 2) AS lost_units_per_stockout_day
FROM d
WHERE period IS NOT NULL
GROUP BY category, period
ORDER BY category, period
"""
con.sql(sql).df()

,category,period,pct_demand_lost,pct_stockout_days,lost_units_per_stockout_day
0,FOODS,1_burn_in,14.53,3.32,12.75
1,FOODS,2_block_a,5.20,1.76,6.25
2,FOODS,3_block_b,4.80,1.52,5.50
3,HOBBIES,1_burn_in,40.96,9.32,6.66
4,HOBBIES,2_block_a,15.85,3.65,3.38
5,HOBBIES,3_block_b,14.64,3.40,2.91
6,HOUSEHOLD,1_burn_in,8.60,1.41,5.90
7,HOUSEHOLD,2_block_a,5.73,0.90,5.08
8,HOUSEHOLD,3_block_b,5.19,0.87,4.62


where in the burn-in does the loss happen? 

In [4]:
sql = f"""
SELECT split_part(item_id, '_', 1) AS category,
       date_diff('day', DATE '{P['burn_in'][0]}', date) // 7 + 1 AS week,
       round(100.0 * sum(lost_sales) / sum(demand), 1) AS pct_demand_lost
FROM read_parquet('../data/processed/sim/daily/*/*.parquet', hive_partitioning=false)
WHERE on_sale AND date BETWEEN '{P['burn_in'][0]}' AND '{P['burn_in'][1]}'
GROUP BY 1, 2
ORDER BY 1, 2
"""
con.sql(sql).df()

,category,week,pct_demand_lost
0,FOODS,1,26.4
1,FOODS,2,16.5
2,FOODS,3,8.0
3,FOODS,4,5.3
4,HOBBIES,1,46.3
5,HOBBIES,2,62.2
6,HOBBIES,3,16.8
7,HOBBIES,4,36.5
8,HOUSEHOLD,1,8.8
9,HOUSEHOLD,2,9.3


In [5]:
sql = f"""
WITH src AS (
    SELECT store_id, item_id, split_part(item_id, '_', 1) AS category,
           date, demand, lost_sales, on_sale
    FROM read_parquet('../data/processed/sim/daily/*/*.parquet', hive_partitioning=false)
),
size AS (
    SELECT store_id, item_id, category,
           ntile(3) OVER (PARTITION BY category ORDER BY avg_d) AS size_group
    FROM (
        SELECT store_id, item_id, category, avg(demand) AS avg_d
        FROM src
        WHERE on_sale AND date BETWEEN '{P['block_a'][0]}' AND '{P['block_a'][1]}'
        GROUP BY 1, 2, 3
    )
)
SELECT category, size_group,
       round(100.0 * sum(lost_sales) / sum(demand), 1) AS pct_lost_first_14_days
FROM src JOIN size USING (store_id, item_id, category)
WHERE on_sale
  AND date BETWEEN DATE '{P['burn_in'][0]}' AND DATE '{P['burn_in'][0]}' + 13
GROUP BY 1, 2
ORDER BY 1, 2
"""
con.sql(sql).df()

,category,size_group,pct_lost_first_14_days
0,FOODS,1,0.3
1,FOODS,2,1.9
2,FOODS,3,27.1
3,HOBBIES,1,0.3
4,HOBBIES,2,1.7
5,HOBBIES,3,60.7
6,HOUSEHOLD,1,0.0
7,HOUSEHOLD,2,0.3
8,HOUSEHOLD,3,13.1
